In [22]:
# Chipathon 2026 - NYCMOS
# This notebook computes the sizing for Gilbert cell mixer switching pair & LO quad based on gm/Id method
# Resource: https://github.com/bmurmann/Chipathon2025/tree/main

from pygmid import Lookup as lk
from scipy.io import loadmat
import numpy as np
import pandas as pd

In [23]:
# Change file depending on where you cloned repo
n = lk('nfet_03v3.mat')
print(n._Lookup__DATA.keys())

dict_keys(['INFO', 'CORNER', 'TEMP', 'VGS', 'VDS', 'VSB', 'L', 'W', 'NFING', 'ID', 'VT', 'GM', 'GMB', 'GDS', 'CGG', 'CGB', 'CGD', 'CGS', 'CDD', 'CSS', 'STH', 'SFL'])


In [50]:
# Target
gain_db = 10
Av = 10**(gain_db/20)

vdd = 3.3 #V
vif_cm = 1.35 #V

itail = 500e-6
id_per_side = itail/2 

rd = (vdd-vif_cm)/(itail/2) # need low common mdoe

gm_id = np.array([4, 6, 8, 10, 12])

gm_required = gm_id * id_per_side
Av = 2/np.pi * gm_required * rd
gain_db = 20*np.log10(Av)

l = 0.28

In [58]:
# Sizing for RF PAIR

Jd = n.lookup('ID_W', GM_ID=gm_id, L = l)
W = id_per_side / Jd
cgg_w = n.lookup('CGG_W', GM_ID=gm_id, L = l)

cgg = cgg_w * W
ft = gm_required / (2 * np.pi * cgg) # Estimate transit freq

vdsat_est = 2/gm_id
itail = 2 * id_per_side

n_options = len(gm_id)

df = pd.DataFrame(
    [gain_db, Av, np.full(n_options, rd), np.full(n_options, gm_required), gm_id, np.full(n_options, id_per_side), np.full(n_options, itail), Jd, W, cgg, ft, vdsat_est],
    index=['gain_db', 'Av', 'Rload', 'gm_required', 'gm/id', 'id_per_side (uA)', 'itail', 'ID_W', 'W_um', 'Cgg', 'ft', 'Vdsat'],
    
)
df

,0,1,2,3,4
gain_db,1.391949e+01,1.744132e+01,1.994009e+01,2.187829e+01,2.346192e+01
Av,4.965634e+00,7.448451e+00,9.931268e+00,1.241409e+01,1.489690e+01
Rload,7.800000e+03,7.800000e+03,7.800000e+03,7.800000e+03,7.800000e+03
gm_required,1.000000e-03,1.500000e-03,2.000000e-03,2.500000e-03,3.000000e-03
gm/id,4.000000e+00,6.000000e+00,8.000000e+00,1.000000e+01,1.200000e+01
id_per_side (uA),2.500000e-04,2.500000e-04,2.500000e-04,2.500000e-04,2.500000e-04
itail,5.000000e-04,5.000000e-04,5.000000e-04,5.000000e-04,5.000000e-04
ID_W,3.564164e-05,1.882823e-05,1.109116e-05,6.847242e-06,4.296085e-06
W_um,7.014267e+00,1.327793e+01,2.254047e+01,3.651105e+01,5.819252e+01
Cgg,8.356598e-15,1.539724e-14,2.543927e-14,4.009870e-14,6.218209e-14


In [57]:
# RF switching quad
# Size based on current it must steer
rf_option = 1; 

gm_id_ss = np.array([8, 10, 12, 15, 18])
l_ss = 0.28

id_ss = id_per_side
jd_ss = n.lookup('ID_W', GM_ID=gm_id_ss, L=l_ss)
w_ss = id_ss / jd_ss

cgg_w_ss = n.lookup('CGG_W', GM_ID=gm_id_ss, L=l_ss)
cg_ss = w_ss * cgg_w_ss

gm_ss = gm_id_ss * id_ss
ft_ss = gm_ss / (2*np.pi*cg_ss)

vdsat_ss = 2/gm_id_ss

df_ss = pd.DataFrame(
    [gm_id_ss, id_ss*np.ones_like(gm_id_ss), w_ss, gm_ss, ft_ss, vdsat_ss, cg_ss],
    index=['gm_id', 'id_ss', 'w_ss', 'gm_ss', 'ft_ss', 'vdsat_ss', 'cg_ss'])
df_ss


,0,1,2,3,4
gm_id,8.000000e+00,1.000000e+01,1.200000e+01,1.500000e+01,1.800000e+01
id_ss,2.500000e-04,2.500000e-04,2.500000e-04,2.500000e-04,2.500000e-04
w_ss,2.254047e+01,3.651105e+01,5.819252e+01,1.191906e+02,2.674417e+02
gm_ss,2.000000e-03,2.500000e-03,3.000000e-03,3.750000e-03,4.500000e-03
ft_ss,1.251254e+10,9.922699e+09,7.678494e+09,4.891709e+09,2.745315e+09
vdsat_ss,2.500000e-01,2.000000e-01,1.666667e-01,1.333333e-01,1.111111e-01
cg_ss,2.543927e-14,4.009870e-14,6.218209e-14,1.220087e-13,2.608798e-13
